# Karting Notebook

October 7, 2025

Data analysis notebook for Karting, with data from Gateway Kartplex outside St Louis in Illinois, and the kart track in Austin at COTA (Circuit of the Americas). Data is collected with the Sensor Logger and Sensor Log iPhone apps, to have recordings of all the sensors on the phone, which typically is GPS at 1hz with approximately +/- 10m and accurate sensor data (such as IMU) at 100hz. The fusion of GPS and IMU data yields a highly accurate race trace.

## Sensor Log

The Sensor Log app can record data and share to a machine in JSON. Sensor Logger, on the other hand, has a sqlite option. Here we will process the Sensor Log data to align with the Sensor Logger app, which is the preferred app going forward.

In [ ]:
file_path = '/Users/nathanverrill/Karting/COTA/SensorLogFiles_nathan-iphone_251007_17-09-39/2025-09-18_17_52_17_nathan-iphone.json'

# Just peek at the first few lines to understand structure
with open(file_path, 'r') as f:
    for i, line in enumerate(f):
        print(line)
        if i > 0:  # Show first 5 lines
            break

In [ ]:
# loads json faster than json library
# polars is pandas alternative that's much faster
# !pip install ijson polars

In [ ]:
import polars as pl
import ijson
import json
import matplotlib.pyplot as plt
%matplotlib inline

# Path to your large JSON file
file_path = '/Users/nathanverrill/Karting/COTA/SensorLogFiles_nathan-iphone_251007_17-09-39/2025-09-18_17_52_17_nathan-iphone.json'

# Method 1: Direct JSON loading with Polars
# This works for smaller files or if you have sufficient memory
def load_with_polars_direct(file_path):
    """
    Load JSON file directly with Polars
    Note: This loads the entire file into memory
    """
    print("Loading JSON with Polars...")
    df = pl.read_json(file_path)
    print(f"Loaded {df.height} records")
    return df

# Method 2: Streaming with ijson then converting to Polars
# More memory efficient for very large files
def load_with_ijson_to_polars(file_path, max_records=None):
    """
    Load a large JSON file using ijson streaming parser and convert to Polars DataFrame
    max_records: optional limit to number of records (for testing)
    """
    print("Starting to stream data with ijson...")
    data = []
    record_count = 0
    
    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            data.append(record)
            record_count += 1
            
            if record_count % 10000 == 0:
                print(f"Loaded {record_count} records...", end='\r')
                
            if max_records and record_count >= max_records:
                break
    
    print(f"\nConverting {record_count} records to Polars DataFrame...")
    df = pl.DataFrame(data)
    return df

# Try direct loading first (may work depending on your memory)
try:
    # For testing, you can limit with n_rows parameter
    # df = pl.read_ndjson(file_path, n_rows=10000)  # Alternative for newline-delimited JSON
    df = load_with_polars_direct(file_path)
except Exception as e:
    print(f"Direct loading failed: {e}")
    print("Falling back to streaming approach...")
    # If direct loading fails due to memory, use the streaming approach
    df = load_with_ijson_to_polars(file_path, max_records=10000)  # Start with a smaller number

# Display basic information
print("\nDataFrame schema:")
print(df.schema)

print("\nFirst few records:")
print(df.head())

### Data exploration for sensor log

In [ ]:
# Convert string columns to appropriate types for numerical columns
# This improves performance and reduces memory usage
numeric_columns = [
    "accelerometerAccelerationX", "accelerometerAccelerationY", "accelerometerAccelerationZ",
    "gyroRotationX", "gyroRotationY", "gyroRotationZ",
    "locationLatitude", "locationLongitude", "locationAltitude", "locationSpeed",
    "motionUserAccelerationX", "motionUserAccelerationY", "motionUserAccelerationZ"
]

# Convert numeric strings to float
for col in numeric_columns:
    if col in df.columns:
        df = df.with_columns(pl.col(col).cast(pl.Float64))

# Convert timestamp columns to datetime
timestamp_cols = [col for col in df.columns if "Time" in col or "time" in col]
for col in timestamp_cols:
    try:
        if "sinceReboot" not in col and "since1970" in col:
            # Unix timestamp
            df = df.with_columns(
                pl.col(col).cast(pl.Float64).cast(pl.Datetime).alias(f"{col}_datetime")
            )
        elif col == "loggingTime":
            # ISO format
            df = df.with_columns(
                pl.col(col).cast(pl.Datetime).alias(f"{col}_datetime")
            )
    except:
        print(f"Could not convert {col}")

# Basic statistics
print("\nBasic statistics:")
print(df.select(numeric_columns).describe())

# Check for missing values
print("\nMissing values per column:")
print(df.null_count())

### Visualize

In [ ]:
# pyarrow love
# !pip install pyarrow

In [ ]:
# Create a time series index
if "loggingTime_datetime" in df.columns:
    time_col = "loggingTime_datetime"
else:
    # Fall back to numeric time
    time_col = "accelerometerTimestamp_sinceReboot"

# Plot accelerometer data
plt.figure(figsize=(12, 6))

# Convert to pandas for easier plotting (Polars to Pandas conversion is efficient)
# Alternatively use the Polars series directly
acc_data = df.select([time_col, "accelerometerAccelerationX", 
                     "accelerometerAccelerationY", "accelerometerAccelerationZ"]).to_pandas()

plt.plot(acc_data[time_col], acc_data["accelerometerAccelerationX"], label='X Acceleration')
plt.plot(acc_data[time_col], acc_data["accelerometerAccelerationY"], label='Y Acceleration')
plt.plot(acc_data[time_col], acc_data["accelerometerAccelerationZ"], label='Z Acceleration')
plt.xlabel('Time')
plt.ylabel('Acceleration (G)')
plt.title('Accelerometer Readings')
plt.legend()
plt.grid(True)
plt.show()

# Plot GPS track if data varies
loc_data = df.select(["locationLongitude", "locationLatitude"]).to_pandas()
if not loc_data["locationLatitude"].eq(loc_data["locationLatitude"].iloc[0]).all():
    plt.figure(figsize=(10, 8))
    plt.scatter(loc_data["locationLongitude"], loc_data["locationLatitude"], 
                c=range(len(loc_data)), cmap='viridis', s=10)
    plt.colorbar(label='Time progression')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('GPS Track')
    plt.axis('equal')
    plt.grid(True)
    plt.show()

In [ ]:
# Example: Calculate G-forces over time (much faster in Polars)
df = df.with_columns([
    (pl.col("accelerometerAccelerationX").pow(2) + 
     pl.col("accelerometerAccelerationY").pow(2) + 
     pl.col("accelerometerAccelerationZ").pow(2)).sqrt().alias("total_g_force")
])

# Find maximum G-force points
max_g_points = df.sort("total_g_force", descending=True).head(10)
print("Maximum G-force points:")
print(max_g_points.select(["loggingTime", "total_g_force", 
                          "locationLatitude", "locationLongitude"]))

In [ ]:
### visualize in kepler

import polars as pl
import ijson
import datetime
from pathlib import Path

# Path to your large JSON file
file_path = '/Users/nathanverrill/Karting/COTA/SensorLogFiles_nathan-iphone_251007_17-09-39/2025-09-18_17_52_17_nathan-iphone.json'
output_path = '/Users/nathanverrill/Karting/COTA/cota_kepler_track.csv'

def process_and_export_for_kepler(file_path, output_path, max_records=None):
    """
    Process JSON file and export selected columns in Kepler.gl compatible format
    """
    print("Starting to stream data with ijson...")
    
    # Create list to store processed records
    processed_records = []
    record_count = 0
    
    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            # Extract required fields
            try:
                # For Kepler.gl, the timestamp needs to be in ISO format
                # If loggingTime is already in ISO format, use that
                if 'loggingTime' in record:
                    timestamp = record['loggingTime']
                else:
                    # Try to construct from Unix timestamp if available
                    ts = float(record.get('locationTimestamp_since1970', 0))
                    if ts > 0:
                        timestamp = datetime.datetime.fromtimestamp(ts).isoformat()
                    else:
                        # Fall back to current processing time
                        timestamp = datetime.datetime.now().isoformat()
                
                # Get lat/lng coordinates
                lat = float(record.get('locationLatitude', 0))
                lng = float(record.get('locationLongitude', 0))
                
                # Get acceleration - include all three dimensions for complete data
                acc_x = float(record.get('accelerometerAccelerationX', 0))
                acc_y = float(record.get('accelerometerAccelerationY', 0))
                acc_z = float(record.get('accelerometerAccelerationZ', 0))
                
                # Calculate magnitude of acceleration (total G-force)
                total_g = (acc_x**2 + acc_y**2 + acc_z**2)**0.5
                
                # Skip records with no valid GPS data
                if lat == 0 and lng == 0:
                    continue
                    
                # Create record with Kepler.gl compatible format
                processed_records.append({
                    'timestamp': timestamp,
                    'latitude': lat,
                    'longitude': lng,
                    'acceleration_x': acc_x,
                    'acceleration_y': acc_y,
                    'acceleration_z': acc_z,
                    'total_g': total_g
                })
                
                record_count += 1
                
                if record_count % 10000 == 0:
                    print(f"Processed {record_count} records...", end='\r')
                    
                if max_records and record_count >= max_records:
                    break
                    
            except (KeyError, ValueError) as e:
                # Skip problematic records
                continue
    
    print(f"\nProcessed {len(processed_records)} valid records with GPS data")
    
    # Convert to Polars DataFrame
    if processed_records:
        df = pl.DataFrame(processed_records)
        
        # Write to CSV
        df.write_csv(output_path)
        print(f"Exported data to {output_path}")
        print(f"File size: {Path(output_path).stat().st_size / (1024*1024):.2f} MB")
        
        return df
    else:
        print("No valid records found to export")
        return None

# Process the file and export CSV for Kepler
# Set max_records=None to process the entire file
df_kepler = process_and_export_for_kepler(file_path, output_path, max_records=None)

# Display sample of the exported data
if df_kepler is not None:
    print("\nSample of exported data:")
    print(df_kepler.head())

### Filter by timestamp

Now have loaded in Kepler and visualized the data to see when meaningful data starts (our notional race). So going to trim the dataset to just that timeframe.

In [ ]:
import polars as pl
import ijson
import datetime
from pathlib import Path

# Path to your large JSON file
file_path = '/Users/nathanverrill/Karting/COTA/SensorLogFiles_nathan-iphone_251007_17-09-39/2025-09-18_17_52_17_nathan-iphone.json'

output_path = '/Users/nathanverrill/Karting/COTA/cota_kepler_track_filtered.csv'


# The timestamp range to filter data (milliseconds since epoch)
START_TIMESTAMP_MS = 1758236524014
END_TIMESTAMP_MS = 1758236765024

# Convert to seconds for comparison with your data
START_TIMESTAMP_SEC = START_TIMESTAMP_MS / 1000.0
END_TIMESTAMP_SEC = END_TIMESTAMP_MS / 1000.0

def export_kepler_compatible_trip(file_path, output_path, start_timestamp_sec, end_timestamp_sec):
    """
    Export data in a format that works reliably with Kepler.gl trip layers
    """
    print(f"Creating Kepler.gl-compatible trip data...")
    
    processed_records = []
    record_count = 0
    filtered_count = 0
    
    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            record_count += 1
            
            if record_count % 10000 == 0:
                print(f"Processed {record_count} records, kept {filtered_count}...", end='\r')
            
            try:
                # Get timestamp for filtering
                timestamp_sec = None
                
                # Try locationTimestamp_since1970 first
                if 'locationTimestamp_since1970' in record:
                    timestamp_sec = float(record['locationTimestamp_since1970'])
                # Try other timestamp fields if needed
                elif 'batteryTimeStamp_since1970' in record:
                    timestamp_sec = float(record['batteryTimeStamp_since1970'])
                elif 'avAudioRecorder_Timestamp_since1970' in record:
                    timestamp_sec = float(record['avAudioRecorder_Timestamp_since1970'])
                elif 'loggingTime' in record:
                    try:
                        dt = datetime.datetime.fromisoformat(record['loggingTime'].replace('Z', '+00:00'))
                        timestamp_sec = dt.timestamp()
                    except:
                        continue
                
                if timestamp_sec is None:
                    continue
                
                # Skip records outside the timestamp range
                if timestamp_sec < start_timestamp_sec or timestamp_sec > end_timestamp_sec:
                    continue
                
                # Get position data
                lat = float(record.get('locationLatitude', 0))
                lng = float(record.get('locationLongitude', 0))
                
                # Skip records with invalid GPS data
                if lat == 0 and lng == 0 or abs(lat) > 90 or abs(lng) > 180:
                    continue
                    
                # Get acceleration data for visualization
                acc_x = float(record.get('accelerometerAccelerationX', 0))
                acc_y = float(record.get('accelerometerAccelerationY', 0))
                acc_z = float(record.get('accelerometerAccelerationZ', 0))
                total_g = (acc_x**2 + acc_y**2 + acc_z**2)**0.5
                
                # Create record with fields in Kepler.gl expected format
                processed_records.append({
                    # Use ISO format timestamps for Kepler compatibility
                    'tpep_pickup_datetime': datetime.datetime.fromtimestamp(timestamp_sec).strftime('%Y-%m-%d %H:%M:%S'),
                    'latitude': lat,
                    'longitude': lng,
                    'total_g': total_g,
                    'acceleration_x': acc_x,
                    'acceleration_y': acc_y,
                    'acceleration_z': acc_z,
                    'trip_id': '1',  # String trip ID works better
                    'speed': float(record.get('locationSpeed', 0)) if float(record.get('locationSpeed', -1)) >= 0 else 0
                })
                
                filtered_count += 1
                    
            except (KeyError, ValueError) as e:
                # Skip problematic records
                continue
    
    print(f"\nProcessed {record_count} total records")
    print(f"Kept {filtered_count} records within timestamp range")
    
    # Convert to Polars DataFrame and ensure proper ordering
    if processed_records:
        df = pl.DataFrame(processed_records)
        
        # Sort by timestamp to ensure proper trip rendering
        df = df.sort("tpep_pickup_datetime")
        
        # Add indices that Kepler sometimes uses
        df = df.with_row_index("index")
        
        # Write to CSV
        df.write_csv(output_path)
        print(f"Exported Kepler-compatible trip data to {output_path}")
        print(f"File size: {Path(output_path).stat().st_size / (1024*1024):.2f} MB")
        
        return df
    else:
        print("No valid records found within timestamp range")
        return None

# Export data in Kepler-compatible format
df_kepler = export_kepler_compatible_trip(file_path, output_path, START_TIMESTAMP_SEC, END_TIMESTAMP_SEC)

## some enhanced stuff for kepler

In [ ]:
def enhanced_karting_data_export(file_path, output_path, start_timestamp_sec, end_timestamp_sec):
    """
    Enhanced export for karting analysis with additional calculated fields
    """
    print(f"Processing karting data between {datetime.datetime.fromtimestamp(start_timestamp_sec).isoformat()} and {datetime.datetime.fromtimestamp(end_timestamp_sec).isoformat()}")
    
    processed_records = []
    record_count = 0
    filtered_count = 0
    last_timestamp = None
    last_lat = None
    last_lng = None
    
    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            record_count += 1
            
            if record_count % 10000 == 0:
                print(f"Processed {record_count} records, kept {filtered_count}...", end='\r')
            
            try:
                # Get timestamp for filtering
                timestamp_sec = None
                for ts_field in ['locationTimestamp_since1970', 'batteryTimeStamp_since1970', 
                                'avAudioRecorder_Timestamp_since1970']:
                    if ts_field in record:
                        timestamp_sec = float(record[ts_field])
                        break
                
                if timestamp_sec is None and 'loggingTime' in record:
                    try:
                        dt = datetime.datetime.fromisoformat(record['loggingTime'].replace('Z', '+00:00'))
                        timestamp_sec = dt.timestamp()
                    except:
                        continue
                
                if timestamp_sec is None:
                    continue
                
                # Skip records outside the timestamp range
                if timestamp_sec < start_timestamp_sec or timestamp_sec > end_timestamp_sec:
                    continue
                
                # Get position data
                lat = float(record.get('locationLatitude', 0))
                lng = float(record.get('locationLongitude', 0))
                
                # Skip records with no valid GPS data
                if lat == 0 and lng == 0 or abs(lat) > 90 or abs(lng) > 180:
                    continue
                
                # Calculate time delta between records
                time_delta = 0
                if last_timestamp is not None:
                    time_delta = timestamp_sec - last_timestamp
                
                # Calculate approximate speed from GPS if locationSpeed is not available
                speed = float(record.get('locationSpeed', -1))
                calculated_speed = -1
                
                if speed < 0 and last_lat is not None and time_delta > 0:
                    # Calculate distance in meters using Haversine formula
                    from math import sin, cos, sqrt, atan2, radians
                    
                    R = 6371000  # Earth radius in meters
                    lat1, lng1 = radians(last_lat), radians(last_lng)
                    lat2, lng2 = radians(lat), radians(lng)
                    
                    dlat = lat2 - lat1
                    dlng = lng2 - lng1
                    
                    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlng/2)**2
                    c = 2 * atan2(sqrt(a), sqrt(1-a))
                    distance = R * c  # Distance in meters
                    
                    # Speed in m/s
                    calculated_speed = distance / time_delta
                    
                    # Convert to km/h for easier reading
                    calculated_speed = calculated_speed * 3.6
                
                # Acceleration data
                acc_x = float(record.get('accelerometerAccelerationX', 0))
                acc_y = float(record.get('accelerometerAccelerationY', 0))
                acc_z = float(record.get('accelerometerAccelerationZ', 0))
                total_g = (acc_x**2 + acc_y**2 + acc_z**2)**0.5
                
                # Get lateral and longitudinal G (assuming device orientation)
                # This is a simplification - proper analysis would account for device orientation
                lateral_g = acc_y  # Side-to-side forces (cornering)
                longitudinal_g = acc_x  # Forward-backward forces (acceleration/braking)
                
                # Create record for Kepler.gl
                processed_records.append({
                    'timestamp': datetime.datetime.fromtimestamp(timestamp_sec).isoformat(),
                    'latitude': lat,
                    'longitude': lng,
                    'altitude': float(record.get('locationAltitude', 0)),
                    'device_speed': speed if speed >= 0 else None,  # Original speed reading
                    'calculated_speed': calculated_speed if calculated_speed >= 0 else None,  # GPS-based speed
                    'acceleration_x': acc_x,
                    'acceleration_y': acc_y, 
                    'acceleration_z': acc_z,
                    'total_g': total_g,
                    'lateral_g': lateral_g,  # Side forces (cornering)
                    'longitudinal_g': longitudinal_g,  # Forward/backward forces
                    'gyro_x': float(record.get('gyroRotationX', 0)),
                    'gyro_y': float(record.get('gyroRotationY', 0)),
                    'gyro_z': float(record.get('gyroRotationZ', 0)),
                    'heading': float(record.get('locationTrueHeading', 0)),
                    'time_delta': time_delta if last_timestamp is not None else 0,
                    'id': 1  # Same ID for all points (for trip layer)
                })
                
                # Save values for next iteration
                last_timestamp = timestamp_sec
                last_lat = lat
                last_lng = lng
                
                filtered_count += 1
                    
            except (KeyError, ValueError) as e:
                # Skip problematic records
                continue
    
    print(f"\nProcessed {record_count} total records")
    print(f"Kept {filtered_count} records within timestamp range")
    
    # Convert to Polars DataFrame
    if processed_records:
        df = pl.DataFrame(processed_records)
        
        # Add a sequential index for animation
        df = df.with_row_index('row_index')
        
        # Write to CSV
        df.write_csv(output_path)
        print(f"Exported karting data to {output_path}")
        print(f"File size: {Path(output_path).stat().st_size / (1024*1024):.2f} MB")
        
        return df
    else:
        print("No valid records found within timestamp range")
        return None

# Uncomment to use the enhanced karting analysis export
enhanced_output_path = '/Users/nathanverrill/Karting/COTA/cota_karting_analysis.csv'
df_enhanced = enhanced_karting_data_export(file_path, enhanced_output_path, START_TIMESTAMP_SEC, END_TIMESTAMP_SEC)

In [ ]:
file_path = '/Users/nathanverrill/Karting/COTA/SensorLogFiles_nathan-iphone_251007_17-09-39/2025-09-18_17_52_17_nathan-iphone.json'


In [ ]:
import polars as pl
import ijson
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline


def load_data_sample(file_path, max_records=10000):
    """
    Load a sample of the data for frequency analysis
    """
    print(f"Loading up to {max_records} records for frequency analysis...")
    
    data = []
    record_count = 0
    
    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            data.append(record)
            record_count += 1
            
            if record_count % 1000 == 0:
                print(f"Loaded {record_count} records...", end='\r')
                
            if record_count >= max_records:
                break
    
    print(f"\nLoaded {record_count} records total")
    return pl.DataFrame(data)

# Load a sample of the data
df = load_data_sample(file_path)

# Verify schema
print("Schema:")
print(df.schema)

# Convert timestamp columns to float for frequency analysis
timestamp_columns = {
    'GPS': 'locationTimestamp_since1970',
    'Accelerometer': 'accelerometerTimestamp_sinceReboot',
    'Gyroscope': 'gyroTimestamp_sinceReboot',
    'Magnetometer': 'magnetometerTimestamp_sinceReboot',
    'Motion': 'motionTimestamp_sinceReboot',
    'Altimeter': 'altimeterTimestamp_sinceReboot',
    'Activity': 'activityTimestamp_sinceReboot'
}

# Calculate frequency for each sensor type
def calculate_update_frequency(df, timestamp_columns):
    """
    Calculate update frequency statistics for each sensor type
    """
    results = {}
    
    for sensor_name, col_name in timestamp_columns.items():
        if col_name in df.columns:
            # Convert to float
            timestamps = df[col_name].cast(pl.Float64).to_numpy()
            
            # Sort timestamps
            timestamps = np.sort(timestamps)
            
            # Calculate time differences between consecutive readings
            time_diffs = np.diff(timestamps)
            
            # Calculate frequencies (1/time_diff)
            frequencies = 1 / time_diffs
            
            # Calculate statistics
            stats = {
                'mean_frequency_hz': np.mean(frequencies),
                'median_frequency_hz': np.median(frequencies),
                'min_frequency_hz': np.min(frequencies),
                'max_frequency_hz': np.max(frequencies),
                'std_frequency_hz': np.std(frequencies),
                'mean_interval_ms': np.mean(time_diffs) * 1000,
                'median_interval_ms': np.median(time_diffs) * 1000,
                'count': len(timestamps),
                'unique_timestamps': len(np.unique(timestamps))
            }
            
            results[sensor_name] = stats
    
    return results

# Check for duplicated timestamps in each sensor
def analyze_timestamp_uniqueness(df, timestamp_columns):
    """
    Analyze how many unique timestamps exist for each sensor type
    """
    results = {}
    
    for sensor_name, col_name in timestamp_columns.items():
        if col_name in df.columns:
            try:
                # Get timestamps
                timestamps = df[col_name].cast(pl.Float64)
                
                # Count total and unique values
                total_count = len(timestamps)
                unique_count = timestamps.n_unique()
                duplicate_ratio = 1 - (unique_count / total_count) if total_count > 0 else 0
                
                results[sensor_name] = {
                    'total_values': total_count,
                    'unique_values': unique_count,
                    'duplicate_ratio': duplicate_ratio,
                    'duplicate_percentage': duplicate_ratio * 100
                }
            except:
                results[sensor_name] = "Error analyzing timestamps"
    
    return results

# Analyze GPS update patterns
def analyze_gps_updates(df):
    """
    Analyze how GPS data is updated compared to other sensors
    """
    # Convert timestamp columns to float
    try:
        df = df.with_columns([
            pl.col('locationTimestamp_since1970').cast(pl.Float64).alias('location_ts'),
            pl.col('accelerometerTimestamp_sinceReboot').cast(pl.Float64).alias('accel_ts')
        ])
        
        # Count unique location timestamps
        unique_location_ts = df['location_ts'].n_unique()
        
        # Group by location timestamp and count accelerometer readings per GPS reading
        result = df.group_by('location_ts').agg(
            pl.count().alias('readings_per_gps'),
            pl.min('accel_ts').alias('min_accel_ts'),
            pl.max('accel_ts').alias('max_accel_ts')
        )
        
        # Calculate statistics
        avg_readings_per_gps = result['readings_per_gps'].mean()
        median_readings_per_gps = result['readings_per_gps'].median()
        max_readings_per_gps = result['readings_per_gps'].max()
        
        # Calculate time span per GPS update
        result = result.with_columns([
            (pl.col('max_accel_ts') - pl.col('min_accel_ts')).alias('time_span')
        ])
        
        avg_time_span = result['time_span'].mean()
        
        return {
            'unique_gps_updates': unique_location_ts,
            'avg_readings_per_gps_update': avg_readings_per_gps,
            'median_readings_per_gps_update': median_readings_per_gps,
            'max_readings_per_gps_update': max_readings_per_gps,
            'avg_time_span_per_gps_update': avg_time_span
        }
    except Exception as e:
        return f"Error analyzing GPS updates: {e}"

# Run the analyses
freq_results = calculate_update_frequency(df, timestamp_columns)
uniqueness_results = analyze_timestamp_uniqueness(df, timestamp_columns)
gps_analysis = analyze_gps_updates(df)

# Print results
print("\n=== SENSOR UPDATE FREQUENCY ANALYSIS ===")
for sensor, stats in freq_results.items():
    print(f"\n{sensor} Sensor:")
    print(f"  Mean frequency: {stats['mean_frequency_hz']:.2f} Hz (update every {stats['mean_interval_ms']:.2f} ms)")
    print(f"  Median frequency: {stats['median_frequency_hz']:.2f} Hz (update every {stats['median_interval_ms']:.2f} ms)")
    print(f"  Range: {stats['min_frequency_hz']:.2f} - {stats['max_frequency_hz']:.2f} Hz")
    print(f"  Sample count: {stats['count']}")
    print(f"  Unique timestamps: {stats['unique_timestamps']} ({(stats['unique_timestamps']/stats['count'])*100:.1f}%)")

print("\n=== TIMESTAMP UNIQUENESS ANALYSIS ===")
for sensor, stats in uniqueness_results.items():
    if isinstance(stats, dict):
        print(f"\n{sensor} Sensor:")
        print(f"  Total readings: {stats['total_values']}")
        print(f"  Unique timestamps: {stats['unique_values']} ({100-stats['duplicate_percentage']:.1f}% unique)")
        print(f"  Duplicate ratio: {stats['duplicate_percentage']:.1f}%")

print("\n=== GPS UPDATE PATTERN ANALYSIS ===")
if isinstance(gps_analysis, dict):
    print(f"Unique GPS updates: {gps_analysis['unique_gps_updates']}")
    print(f"Average readings per GPS update: {gps_analysis['avg_readings_per_gps_update']:.2f}")
    print(f"Median readings per GPS update: {gps_analysis['median_readings_per_gps_update']}")
    print(f"Maximum readings per GPS update: {gps_analysis['max_readings_per_gps_update']}")
    print(f"Average time span per GPS update: {gps_analysis['avg_time_span_per_gps_update']:.4f} seconds")
else:
    print(gps_analysis)

# Visualize sensor update frequencies
plt.figure(figsize=(12, 8))
labels = []
means = []
medians = []

for sensor, stats in freq_results.items():
    labels.append(sensor)
    means.append(stats['mean_frequency_hz'])
    medians.append(stats['median_frequency_hz'])

x = range(len(labels))
width = 0.35

plt.bar([i - width/2 for i in x], means, width, label='Mean Frequency (Hz)')
plt.bar([i + width/2 for i in x], medians, width, label='Median Frequency (Hz)')
plt.xlabel('Sensor Type')
plt.ylabel('Frequency (Hz)')
plt.title('Sensor Update Frequencies')
plt.xticks(x, labels, rotation=45)
plt.legend()
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add frequency values on top of bars
for i, v in enumerate(means):
    plt.text(i - width/2, v + 0.5, f"{v:.1f}", ha='center')
for i, v in enumerate(medians):
    plt.text(i + width/2, v + 0.5, f"{v:.1f}", ha='center')

plt.show()

# Plot GPS updates vs. accelerometer readings
if isinstance(gps_analysis, dict) and 'locationTimestamp_since1970' in df.columns and 'accelerometerTimestamp_sinceReboot' in df.columns:
    try:
        # Create sorted timestamps
        loc_ts = df['locationTimestamp_since1970'].cast(pl.Float64).sort()
        accel_ts = df['accelerometerTimestamp_sinceReboot'].cast(pl.Float64).sort()
        
        # Normalize timestamps to start from 0
        loc_ts_norm = (loc_ts - loc_ts.min()).to_numpy()
        accel_ts_norm = (accel_ts - accel_ts.min()).to_numpy()
        
        plt.figure(figsize=(14, 6))
        
        # Plot cumulative counts
        plt.plot(loc_ts_norm, range(len(loc_ts_norm)), 'b-', linewidth=2, label='GPS Updates')
        plt.plot(accel_ts_norm, range(len(accel_ts_norm)), 'r-', linewidth=1, label='Accelerometer Readings')
        
        plt.xlabel('Time (seconds since start)')
        plt.ylabel('Cumulative Number of Readings')
        plt.title('GPS Updates vs. Accelerometer Readings Over Time')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.show()
    except Exception as e:
        print(f"Error creating GPS vs. accelerometer plot: {e}")

In [ ]:
import polars as pl
import ijson
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
from pathlib import Path
%matplotlib inline

# Path to your large JSON file
output_path = '/Users/nathanverrill/Karting/COTA/cota_fused_location.csv'

# The timestamp range to filter data (milliseconds since epoch)
START_TIMESTAMP_MS = 1758236524014
END_TIMESTAMP_MS = 1758236765024
START_TIMESTAMP_SEC = START_TIMESTAMP_MS / 1000.0
END_TIMESTAMP_SEC = END_TIMESTAMP_MS / 1000.0

def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance in meters between two points 
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371000  # Radius of earth in meters
    return c * r

def bearing(lat1, lon1, lat2, lon2):
    """
    Calculate the bearing between two points in radians
    """
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlon = lon2 - lon1
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    
    bearing = np.arctan2(y, x)
    return bearing

def new_position(lat, lon, distance, bearing):
    """
    Calculate a new position given a starting point, distance, and bearing
    """
    lat, lon, bearing = map(np.radians, [lat, lon, bearing])
    
    R = 6371000  # Earth radius in meters
    
    lat2 = np.arcsin(np.sin(lat) * np.cos(distance/R) + 
                     np.cos(lat) * np.sin(distance/R) * np.cos(bearing))
    
    lon2 = lon + np.arctan2(np.sin(bearing) * np.sin(distance/R) * np.cos(lat),
                           np.cos(distance/R) - np.sin(lat) * np.sin(lat2))
    
    return np.degrees(lat2), np.degrees(lon2)

def process_and_fuse_location(file_path, output_path, start_timestamp_sec, end_timestamp_sec):
    """
    Process JSON file, apply sensor fusion for location data, and export
    """
    print(f"Loading and fusing location data between timestamps...")
    
    # Lists to store processed data
    gps_records = []  # Original GPS records
    all_records = []  # All sensor records
    
    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            try:
                # Get timestamps
                loc_timestamp = float(record.get('locationTimestamp_since1970', 0))
                accel_timestamp = float(record.get('accelerometerTimestamp_sinceReboot', 0))
                
                # Skip records outside the timestamp range
                if loc_timestamp > 0 and (loc_timestamp < start_timestamp_sec or loc_timestamp > end_timestamp_sec):
                    continue
                
                # Get location data
                lat = float(record.get('locationLatitude', 0))
                lon = float(record.get('locationLongitude', 0))
                speed = float(record.get('locationSpeed', -1))
                course = float(record.get('locationCourse', -1))
                
                # Get motion data
                acc_x = float(record.get('accelerometerAccelerationX', 0))
                acc_y = float(record.get('accelerometerAccelerationY', 0))
                acc_z = float(record.get('accelerometerAccelerationZ', 0))
                
                # Get rotation data
                gyro_x = float(record.get('gyroRotationX', 0))
                gyro_y = float(record.get('gyroRotationY', 0))
                gyro_z = float(record.get('gyroRotationZ', 0))
                
                # If this is a valid GPS record, store separately
                if lat != 0 and lon != 0 and abs(lat) <= 90 and abs(lon) <= 180 and loc_timestamp > 0:
                    gps_records.append({
                        'timestamp': loc_timestamp,
                        'latitude': lat,
                        'longitude': lon,
                        'speed': speed if speed >= 0 else 0,
                        'course': course if course >= 0 else 0,
                        'is_original_gps': True
                    })
                
                # Store all records for sensor fusion
                all_records.append({
                    'timestamp': loc_timestamp if loc_timestamp > 0 else 0,
                    'accel_timestamp': accel_timestamp,
                    'latitude': lat,
                    'longitude': lon,
                    'speed': speed if speed >= 0 else 0,
                    'course': course if course >= 0 else 0,
                    'acc_x': acc_x,
                    'acc_y': acc_y,
                    'acc_z': acc_z,
                    'gyro_x': gyro_x, 
                    'gyro_y': gyro_y,
                    'gyro_z': gyro_z,
                    'has_valid_gps': lat != 0 and lon != 0 and abs(lat) <= 90 and abs(lon) <= 180
                })
                
            except (KeyError, ValueError) as e:
                # Skip problematic records
                continue
    
    print(f"Loaded {len(all_records)} total records")
    print(f"Found {len(gps_records)} GPS records")
    
    # Convert to Polars DataFrames
    if gps_records and all_records:
        gps_df = pl.DataFrame(gps_records)
        all_df = pl.DataFrame(all_records)
        
        # Sort by timestamp
        gps_df = gps_df.sort('timestamp')
        all_df = all_df.sort('accel_timestamp')
        
        print("Performing location fusion...")
        
        # Convert to pandas for easier processing with NumPy
        gps_pd = gps_df.to_pandas()
        all_pd = all_df.to_pandas()
        
        # Initialize arrays for fused positions
        fused_positions = []
        
        # Get the first GPS position
        last_gps_idx = 0
        next_gps_idx = 1
        
        # Process each record and estimate position
        for idx, row in all_pd.iterrows():
            current_time = row['accel_timestamp']
            
            # Find the appropriate GPS bracket for this timestamp
            while next_gps_idx < len(gps_pd) and gps_pd.iloc[next_gps_idx]['timestamp'] <= current_time:
                last_gps_idx = next_gps_idx
                next_gps_idx += 1
                
            # If we have valid GPS data
            if row['has_valid_gps']:
                # Use the actual GPS position
                fused_positions.append({
                    'timestamp': current_time,
                    'latitude': row['latitude'],
                    'longitude': row['longitude'],
                    'is_fused': False,
                    'total_g': np.sqrt(row['acc_x']**2 + row['acc_y']**2 + row['acc_z']**2)
                })
            elif last_gps_idx < len(gps_pd) and next_gps_idx < len(gps_pd):
                # Perform linear interpolation between GPS points
                gps1 = gps_pd.iloc[last_gps_idx]
                gps2 = gps_pd.iloc[next_gps_idx]
                
                # Time ratios for interpolation
                t_diff = gps2['timestamp'] - gps1['timestamp']
                if t_diff > 0:
                    ratio = (current_time - gps1['timestamp']) / t_diff
                    
                    # Simple linear interpolation
                    lat = gps1['latitude'] + ratio * (gps2['latitude'] - gps1['latitude'])
                    lon = gps1['longitude'] + ratio * (gps2['longitude'] - gps1['longitude'])
                    
                    # Store interpolated position
                    fused_positions.append({
                        'timestamp': current_time,
                        'latitude': lat,
                        'longitude': lon,
                        'is_fused': True,
                        'total_g': np.sqrt(row['acc_x']**2 + row['acc_y']**2 + row['acc_z']**2)
                    })
                else:
                    # If GPS timestamps are identical, use the first position
                    fused_positions.append({
                        'timestamp': current_time,
                        'latitude': gps1['latitude'],
                        'longitude': gps1['longitude'],
                        'is_fused': True,
                        'total_g': np.sqrt(row['acc_x']**2 + row['acc_y']**2 + row['acc_z']**2)
                    })
            elif last_gps_idx < len(gps_pd):
                # After the last GPS point, use the last known position
                last_gps = gps_pd.iloc[last_gps_idx]
                fused_positions.append({
                    'timestamp': current_time,
                    'latitude': last_gps['latitude'],
                    'longitude': last_gps['longitude'],
                    'is_fused': True,
                    'total_g': np.sqrt(row['acc_x']**2 + row['acc_y']**2 + row['acc_z']**2)
                })
            else:
                # Before the first GPS point, skip
                continue
        
        # Convert fused positions to DataFrame
        fused_df = pl.DataFrame(fused_positions)
        
        # Write to CSV
        fused_df.write_csv(output_path)
        print(f"Exported fused location data to {output_path}")
        print(f"File size: {Path(output_path).stat().st_size / (1024*1024):.2f} MB")
        
        # Visualize original GPS vs fused positions
        plt.figure(figsize=(12, 10))
        
        # Plot original GPS points
        gps_data = gps_df.to_pandas()
        plt.scatter(gps_data['longitude'], gps_data['latitude'], 
                   color='blue', s=30, label='Original GPS', alpha=0.7)
        
        # Plot fused positions (sample for clarity)
        fused_data = fused_df.to_pandas()
        sample_size = min(1000, len(fused_data))
        sample = fused_data.sample(sample_size)
        plt.scatter(sample['longitude'], sample['latitude'], 
                   color='red', s=10, label='Fused Positions', alpha=0.3)
        
        plt.title('Original GPS vs Fused Positions')
        plt.xlabel('Longitude')
        plt.ylabel('Latitude')
        plt.legend()
        plt.axis('equal')
        plt.grid(True)
        plt.show()
        
        return fused_df, gps_df
    
    else:
        print("No valid records found for fusion")
        return None, None

# Run the location fusion
fused_df, gps_df = process_and_fuse_location(file_path, output_path, START_TIMESTAMP_SEC, END_TIMESTAMP_SEC)

In [ ]:
def process_with_kalman_filter(file_path, output_path, start_timestamp_sec, end_timestamp_sec):
    """
    Process JSON file using Kalman filter for optimal sensor fusion
    """
    from filterpy.kalman import KalmanFilter
    from filterpy.common import Q_discrete_white_noise
    
    print(f"Implementing Kalman filter fusion between timestamps...")
    
    # First pass: collect GPS data
    gps_records = []
    all_records = []
    
    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            try:
                # Get timestamps
                loc_timestamp = float(record.get('locationTimestamp_since1970', 0))
                accel_timestamp = float(record.get('accelerometerTimestamp_sinceReboot', 0))
                
                # Skip records outside the timestamp range
                if loc_timestamp > 0 and (loc_timestamp < start_timestamp_sec or loc_timestamp > end_timestamp_sec):
                    continue
                
                # Get location data
                lat = float(record.get('locationLatitude', 0))
                lon = float(record.get('locationLongitude', 0))
                speed = float(record.get('locationSpeed', -1))
                course = float(record.get('locationCourse', -1))
                horizontal_accuracy = float(record.get('locationHorizontalAccuracy', 10))
                
                # Get motion data
                acc_x = float(record.get('accelerometerAccelerationX', 0))
                acc_y = float(record.get('accelerometerAccelerationY', 0))
                acc_z = float(record.get('accelerometerAccelerationZ', 0))
                
                # Store valid GPS points separately
                if lat != 0 and lon != 0 and abs(lat) <= 90 and abs(lon) <= 180 and loc_timestamp > 0:
                    gps_records.append({
                        'timestamp': loc_timestamp,
                        'latitude': lat,
                        'longitude': lon,
                        'speed': speed if speed >= 0 else 0,
                        'course': course if course >= 0 else 0,
                        'accuracy': horizontal_accuracy
                    })
                
                # Store all records with accelerometer data for fusion
                all_records.append({
                    'timestamp': accel_timestamp,
                    'gps_timestamp': loc_timestamp if loc_timestamp > 0 else None,
                    'latitude': lat if lat != 0 and abs(lat) <= 90 else None,
                    'longitude': lon if lon != 0 and abs(lon) <= 180 else None,
                    'speed': speed if speed >= 0 else None,
                    'course': course if course >= 0 else None,
                    'acc_x': acc_x,
                    'acc_y': acc_y,
                    'acc_z': acc_z,
                    'has_valid_gps': lat != 0 and lon != 0 and abs(lat) <= 90 and abs(lon) <= 180 and loc_timestamp > 0
                })
                
            except (KeyError, ValueError):
                continue
    
    print(f"Collected {len(gps_records)} GPS records and {len(all_records)} total records")
    
    if not gps_records or not all_records:
        print("Insufficient data for Kalman filtering")
        return None
    
    # Convert to DataFrames and sort by timestamp
    gps_df = pl.DataFrame(gps_records).sort('timestamp')
    all_df = pl.DataFrame(all_records).sort('timestamp')
    
    # Convert to pandas for easier processing with Kalman filter
    gps_pd = gps_df.to_pandas()
    all_pd = all_df.to_pandas()
    
    # Initialize Kalman filter
    # State vector: [lat, lon, lat_vel, lon_vel]
    kf = KalmanFilter(dim_x=4, dim_z=2)
    
    # Initial state from first GPS reading
    initial_gps = gps_pd.iloc[0]
    kf.x = np.array([
        initial_gps['latitude'],
        initial_gps['longitude'],
        0,  # Initial latitude velocity
        0   # Initial longitude velocity
    ])
    
    # State transition matrix (prediction step)
    dt = 0.01  # 10ms timestep, adjust based on sensor frequency
    kf.F = np.array([
        [1, 0, dt, 0],
        [0, 1, 0, dt],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ])
    
    # Measurement matrix (only measuring lat/lon)
    kf.H = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0]
    ])
    
    # Measurement noise covariance
    # Higher values mean less trust in GPS measurements
    initial_accuracy = initial_gps['accuracy']
    kf.R = np.array([
        [initial_accuracy**2, 0],
        [0, initial_accuracy**2]
    ])
    
    # Process noise covariance
    # Q represents how much we expect the position to change between measurements
    q = 0.01  # Process noise
    kf.Q = Q_discrete_white_noise(dim=4, dt=dt, var=q)
    
    # Initial state covariance
    kf.P = np.array([
        [initial_accuracy**2, 0, 0, 0],
        [0, initial_accuracy**2, 0, 0],
        [0, 0, 10, 0],
        [0, 0, 0, 10]
    ])
    
    # Process each record and apply Kalman filtering
    kalman_positions = []
    last_timestamp = None
    last_gps_idx = 0
    
    # Extract coordinates for distance/speed calculations
    lat_meters_per_degree = 111320  # Approximate meters per degree of latitude
    lon_meters_per_degree = 111320 * np.cos(np.radians(initial_gps['latitude']))  # Depends on latitude
    
    for idx, row in all_pd.iterrows():
        current_time = row['timestamp']
        
        # Calculate time delta for prediction step
        if last_timestamp is not None:
            dt = current_time - last_timestamp
        else:
            dt = 0.01  # Default time step
        
        # Update state transition matrix with current dt
        kf.F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ])
        
        # Update process noise with current dt
        kf.Q = Q_discrete_white_noise(dim=4, dt=dt, var=q)
        
        # Prediction step
        kf.predict()
        
        # If this record has valid GPS, use it for correction
        if row['has_valid_gps']:
            # Find corresponding GPS record
            while last_gps_idx < len(gps_pd) and gps_pd.iloc[last_gps_idx]['timestamp'] < row['gps_timestamp']:
                last_gps_idx += 1
            
            if last_gps_idx < len(gps_pd) and gps_pd.iloc[last_gps_idx]['timestamp'] == row['gps_timestamp']:
                gps_record = gps_pd.iloc[last_gps_idx]
                
                # Update measurement noise based on GPS accuracy
                accuracy = gps_record['accuracy']
                kf.R = np.array([
                    [accuracy**2, 0],
                    [0, accuracy**2]
                ])
                
                # Update step
                z = np.array([row['latitude'], row['longitude']])
                kf.update(z)
        
        # Store filtered position
        filtered_lat, filtered_lon = kf.x[0], kf.x[1]
        lat_vel, lon_vel = kf.x[2], kf.x[3]
        
        # Calculate speed in m/s from velocity components
        speed_lat = lat_vel * lat_meters_per_degree
        speed_lon = lon_vel * lon_meters_per_degree
        speed = np.sqrt(speed_lat**2 + speed_lon**2)
        
        # Calculate course/heading in degrees
        course = np.degrees(np.arctan2(speed_lon, speed_lat)) % 360
        
        kalman_positions.append({
            'timestamp': current_time,
            'original_latitude': row['latitude'],
            'original_longitude': row['longitude'],
            'filtered_latitude': filtered_lat,
            'filtered_longitude': filtered_lon,
            'speed': speed,
            'course': course,
            'total_g': np.sqrt(row['acc_x']**2 + row['acc_y']**2 + row['acc_z']**2),
            'is_gps_update': row['has_valid_gps']
        })
        
        last_timestamp = current_time
    
    # Convert filtered positions to DataFrame
    filtered_df = pl.DataFrame(kalman_positions)
    
    # Write to CSV
    filtered_df.write_csv(output_path)
    print(f"Exported Kalman-filtered location data to {output_path}")
    print(f"File size: {Path(output_path).stat().st_size / (1024*1024):.2f} MB")
    
    # Visualize original vs filtered positions
    plt.figure(figsize=(12, 10))
    
    # Get data points where original GPS was available
    gps_points = filtered_df.filter(pl.col('is_gps_update') == True).to_pandas()
    
    # Plot original GPS points
    plt.scatter(gps_points['original_longitude'], gps_points['original_latitude'], 
               color='blue', s=30, label='Original GPS', alpha=0.7)
    
    # Plot filtered positions (sample for clarity)
    filtered_data = filtered_df.to_pandas()
    sample_size = min(2000, len(filtered_data))
    sample = filtered_data.sample(sample_size)
    plt.scatter(sample['filtered_longitude'], sample['filtered_longitude'], 
               color='red', s=10, label='Kalman Filtered', alpha=0.3)
    
    plt.title('Original GPS vs Kalman-Filtered Positions')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.legend()
    plt.axis('equal')
    plt.grid(True)
    plt.show()
    
    # Plot speed profile
    plt.figure(figsize=(12, 6))
    plt.plot(filtered_data['timestamp'], filtered_data['speed'], 'g-', alpha=0.7)
    plt.title('Kalman-Filtered Speed Profile')
    plt.xlabel('Time')
    plt.ylabel('Speed (m/s)')
    plt.grid(True)
    plt.show()
    
    return filtered_df

# Run the Kalman filter fusion
kalman_output_path = '/Users/nathanverrill/Karting/COTA/cota_kalman_filtered.csv'
filtered_df = process_with_kalman_filter(file_path, kalman_output_path, START_TIMESTAMP_SEC, END_TIMESTAMP_SEC)

In [ ]:
def process_with_dead_reckoning(file_path, output_path, start_timestamp_sec, end_timestamp_sec):
    """
    Process JSON file using dead reckoning between GPS updates
    """
    print(f"Implementing dead reckoning fusion between timestamps...")
    
    # First pass: collect all relevant data
    records = []
    
    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            try:
                # Get timestamps
                loc_timestamp = float(record.get('locationTimestamp_since1970', 0))
                accel_timestamp = float(record.get('accelerometerTimestamp_sinceReboot', 0))
                
                # Skip records outside the timestamp range
                if loc_timestamp > 0 and (loc_timestamp < start_timestamp_sec or loc_timestamp > end_timestamp_sec):
                    continue
                
                # Get all relevant data
                data = {
                    'timestamp': accel_timestamp,
                    'gps_timestamp': loc_timestamp if loc_timestamp > 0 else None,
                    'latitude': float(record.get('locationLatitude', 0)),
                    'longitude': float(record.get('locationLongitude', 0)),
                    'altitude': float(record.get('locationAltitude', 0)),
                    'speed': float(record.get('locationSpeed', -1)),
                    'course': float(record.get('locationCourse', -1)),
                    'true_heading': float(record.get('locationTrueHeading', -1)),
                    'horizontal_accuracy': float(record.get('locationHorizontalAccuracy', 10)),
                    'acc_x': float(record.get('accelerometerAccelerationX', 0)),
                    'acc_y': float(record.get('accelerometerAccelerationY', 0)),
                    'acc_z': float(record.get('accelerometerAccelerationZ', 0)),
                    'gyro_x': float(record.get('gyroRotationX', 0)),
                    'gyro_y': float(record.get('gyroRotationY', 0)),
                    'gyro_z': float(record.get('gyroRotationZ', 0)),
                    # Device orientation and motion data is extremely useful for dead reckoning
                    'motion_heading': float(record.get('motionHeading', -1)),
                    'motion_roll': float(record.get('motionRoll', 0)),
                    'motion_pitch': float(record.get('motionPitch', 0)),
                    'motion_yaw': float(record.get('motionYaw', 0))
                }
                
                # Flag valid GPS data
                data['has_valid_gps'] = (data['latitude'] != 0 and 
                                         data['longitude'] != 0 and 
                                         abs(data['latitude']) <= 90 and 
                                         abs(data['longitude']) <= 180 and 
                                         loc_timestamp > 0)
                
                records.append(data)
                
            except (KeyError, ValueError) as e:
                continue
    
    print(f"Collected {len(records)} total records")
    
    if not records:
        print("No valid records found")
        return None
    
    # Convert to DataFrame and sort by timestamp
    df = pl.DataFrame(records).sort('timestamp')
    
    # Convert to pandas for easier processing
    df_pd = df.to_pandas()
    
    # Perform dead reckoning
    dr_positions = []
    last_valid_gps = None
    last_valid_gps_time = None
    last_timestamp = None
    integrated_course = None
    
    # Get Earth radius at this latitude (for more accurate calculations)
    mean_lat = df_pd['latitude'].mean()
    lat_meters_per_degree = 111320  # Approximate meters per degree of latitude
    lon_meters_per_degree = 111320 * np.cos(np.radians(mean_lat))  # Depends on latitude
    
    for idx, row in df_pd.iterrows():
        current_time = row['timestamp']
        
        if row['has_valid_gps']:
            # If valid GPS, use it directly
            dr_positions.append({
                'timestamp': current_time,
                'latitude': row['latitude'],
                'longitude': row['longitude'],
                'altitude': row['altitude'],
                'speed': max(0, row['speed']),
                'course': row['true_heading'] if row['true_heading'] >= 0 else 
                          (row['course'] if row['course'] >= 0 else
                           (row['motion_heading'] if row['motion_heading'] >= 0 else 0)),
                'total_g': np.sqrt(row['acc_x']**2 + row['acc_y']**2 + row['acc_z']**2),
                'source': 'GPS',
                'accuracy': row['horizontal_accuracy']
            })
            
            # Save this as reference for dead reckoning
            last_valid_gps = row
            last_valid_gps_time = current_time
            
            # Reset integrated course to match GPS course
            if row['true_heading'] >= 0:
                integrated_course = row['true_heading']
            elif row['course'] >= 0:
                integrated_course = row['course']
            elif row['motion_heading'] >= 0:
                integrated_course = row['motion_heading']
        
        elif last_valid_gps is not None:
            # Calculate time since last GPS update
            dt = current_time - last_timestamp if last_timestamp is not None else 0.01
            
            # Integrate gyro_z to update heading/course
            if integrated_course is not None:
                # Convert gyro_z from radians/s to degrees/s (approx 57.3 deg/rad)
                degrees_per_second = row['gyro_z'] * 57.3
                integrated_course = (integrated_course + degrees_per_second * dt) % 360
            else:
                # Use any available heading information
                if row['true_heading'] >= 0:
                    integrated_course = row['true_heading']
                elif row['course'] >= 0:
                    integrated_course = row['course']
                elif row['motion_heading'] >= 0:
                    integrated_course = row['motion_heading']
                else:
                    integrated_course = 0  # Default heading
            
            # Estimate current speed
            if row['speed'] >= 0:
                current_speed = row['speed']
            else:
                # Use accelerometer data to update speed
                last_pos = dr_positions[-1]
                last_speed = last_pos['speed']
                
                # Get acceleration in the forward direction
                # This requires knowing device orientation relative to vehicle
                # Simplified approach: use total acceleration magnitude
                accel_magnitude = np.sqrt(row['acc_x']**2 + row['acc_y']**2 + row['acc_z']**2)
                # Subtract 1g (gravity)
                accel_magnitude = max(0, accel_magnitude - 1.0)
                
                # Assume acceleration is in the direction of travel
                # Convert G to m/s²
                accel_ms2 = accel_magnitude * 9.81
                
                # Update speed: v = u + at
                current_speed = max(0, last_speed + accel_ms2 * dt)
            
            # Calculate distance traveled
            distance = current_speed * dt
            
            # Get last position
            last_pos = dr_positions[-1]
            last_lat = last_pos['latitude']
            last_lon = last_pos['longitude']
            
            # Calculate new position based on course and distance
            # Convert course from degrees to radians
            course_rad = np.radians(integrated_course)
            
            # Calculate position offsets
            lat_offset = distance * np.cos(course_rad) / lat_meters_per_degree
            lon_offset = distance * np.sin(course_rad) / lon_meters_per_degree
            
            # Calculate new position
            new_lat = last_lat + lat_offset
            new_lon = last_lon + lon_offset
            
            # Calculate accuracy based on time since last GPS fix
            time_since_gps = current_time - last_valid_gps_time
            # Accuracy decreases over time (simplified model)
            accuracy_degradation = 5 * time_since_gps  # 5 meters per second degradation
            base_accuracy = last_valid_gps['horizontal_accuracy']
            current_accuracy = base_accuracy + accuracy_degradation
            
            dr_positions.append({
                'timestamp': current_time,
                'latitude': new_lat,
                'longitude': new_lon,
                'altitude': last_pos['altitude'],  # Maintain last altitude
                'speed': current_speed,
                'course': integrated_course,
                'total_g': np.sqrt(row['acc_x']**2 + row['acc_y']**2 + row['acc_z']**2),
                'source': 'Dead Reckoning',
                'accuracy': current_accuracy,
                'time_since_gps': time_since_gps
            })
        
        last_timestamp = current_time
    
    # Convert dead reckoning positions to DataFrame
    dr_df = pl.DataFrame(dr_positions)
    
    # Write to CSV
    dr_df.write_csv(output_path)
    print(f"Exported dead reckoning location data to {output_path}")
    print(f"File size: {Path(output_path).stat().st_size / (1024*1024):.2f} MB")
    
    # Visualize dead reckoning results
    plt.figure(figsize=(12, 10))
    
    # Convert to pandas for plotting
    dr_pd = dr_df.to_pandas()
    
    # Plot GPS points
    gps_points = dr_pd[dr_pd['source'] == 'GPS']
    plt.scatter(gps_points['longitude'], gps_points['latitude'], 
               color='blue', s=30, label='GPS', alpha=0.7)
    
    # Plot dead reckoning points
    dr_points = dr_pd[dr_pd['source'] == 'Dead Reckoning']
    # Sample if too many points
    if len(dr_points) > 2000:
        dr_points = dr_points.sample(2000)
    plt.scatter(dr_points['longitude'], dr_points['latitude'], 
               color='red', s=10, label='Dead Reckoning', alpha=0.3)
    
    plt.title('GPS vs Dead Reckoning Positions')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.legend()
    plt.axis('equal')
    plt.grid(True)
    plt.show()
    
    # Plot speed profile
    plt.figure(figsize=(12, 6))
    plt.plot(dr_pd['timestamp'], dr_pd['speed'], 'g-', alpha=0.7)
    plt.title('Speed Profile')
    plt.xlabel('Time')
    plt.ylabel('Speed (m/s)')
    plt.grid(True)
    plt.show()
    
    return dr_df

# Run the dead reckoning fusion
dr_output_path = '/Users/nathanverrill/Karting/COTA/cota_dead_reckoning.csv'
dr_df = process_with_dead_reckoning(file_path, dr_output_path, START_TIMESTAMP_SEC, END_TIMESTAMP_SEC)

In [ ]:
def process_with_map_matching(file_path, output_path, start_timestamp_sec, end_timestamp_sec):
    """
    Process JSON file with sensor fusion and map matching to COTA track
    """
    print(f"Implementing map matching fusion...")
    
    # COTA track coordinates (simplified for example)
    # In a real implementation, you would use a detailed track map with waypoints
    # This is a rough approximation of COTA's main waypoints
    cota_waypoints = np.array([
        [30.134735, -97.634178],  # Start/finish line
        [30.136551, -97.635058],  # Turn 1 apex
        [30.136926, -97.636303],  # Turn 2
        [30.136555, -97.636797],  # Turn 3
        [30.135521, -97.637216],  # Turn 4
        [30.134571, -97.637966],  # Turn 5
        [30.133781, -97.638669],  # Turn 6
        [30.133185, -97.639570],  # Turn 7
        [30.132499, -97.640444],  # Turn 8
        [30.131754, -97.641013],  # Turn 9
        [30.130939, -97.640994],  # Turn 10
        [30.130083, -97.640260],  # Turn 11
        [30.129553, -97.639326],  # Exit Turn 11
        [30.128775, -97.636937],  # Back straight
        [30.128463, -97.634582],  # Approaching Turn 12
        [30.128668, -97.632853],  # Turn 12 apex
        [30.129197, -97.631825],  # Turn 13
        [30.129899, -97.631030],  # Turn 14
        [30.130594, -97.630739],  # Turn 15
        [30.131498, -97.630471],  # Turn 16
        [30.132485, -97.630703],  # Turn 17
        [30.133295, -97.631369],  # Turn 18
        [30.134020, -97.632142],  # Turn 19
        [30.134630, -97.633058],  # Turn 20
        [30.134735, -97.634178],  # Back to start
    ])
    
    # First pass: collect data
    # (Same as in previous functions)
    records = []
    # ...collecting data code as before...
    
    # First process with Kalman or dead reckoning as before
    # Then apply map matching to snap positions to track
    
    # Example of map matching function:
    def match_to_track(lat, lon, track_waypoints):
        """
        Match a position to the nearest point on the track
        """
        point = np.array([lat, lon])
        distances = np.sum((track_waypoints - point)**2, axis=1)
        nearest_idx = np.argmin(distances)
        
        # Get nearest waypoint
        nearest_point = track_waypoints[nearest_idx]
        
        # If we have a high density track map, we'd return the nearest point
        # For a sparse map like our example, we'd interpolate between waypoints
        next_idx = (nearest_idx + 1) % len(track_waypoints)
        prev_idx = (nearest_idx - 1) % len(track_waypoints)
        
        next_point = track_waypoints[next_idx]
        prev_point = track_waypoints[prev_idx]
        
        # Find the closest line segment (between waypoints)
        # and project the point onto that segment
        
        # Simple implementation: just return the nearest waypoint
        # In a full implementation, you'd project onto the nearest track segment
        return nearest_point[0], nearest_point[1]
    
    # This function would be applied to the fused positions
    # to snap them to the track
    
    # For a production implementation, look into map matching algorithms
    # like Hidden Markov Models (HMM) for map matching

In [ ]:
import pandas as pd

def process_karting_data_fusion(file_path, output_path, start_timestamp_sec, end_timestamp_sec):
    """
    Practical sensor fusion approach for karting data
    Combines interpolation, dead reckoning, and smoothing
    """
    import polars as pl
    import ijson
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy import interpolate
    from scipy.signal import savgol_filter
    %matplotlib inline
    
    print("Implementing hybrid fusion for karting data...")
    
    # First pass: collect all records
    records = []
    
    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            try:
                # Get timestamps
                loc_timestamp = float(record.get('locationTimestamp_since1970', 0))
                accel_timestamp = float(record.get('accelerometerTimestamp_sinceReboot', 0))
                
                # Skip records outside the timestamp range
                if loc_timestamp > 0 and (loc_timestamp < start_timestamp_sec or loc_timestamp > end_timestamp_sec):
                    continue
                
                # Get all relevant data
                data = {
                    'timestamp': accel_timestamp,
                    'gps_timestamp': loc_timestamp if loc_timestamp > 0 else None,
                    'latitude': float(record.get('locationLatitude', 0)),
                    'longitude': float(record.get('locationLongitude', 0)),
                    'altitude': float(record.get('locationAltitude', 0)),
                    'speed': float(record.get('locationSpeed', -1)),
                    'course': float(record.get('locationTrueHeading', -1)),
                    'horizontal_accuracy': float(record.get('locationHorizontalAccuracy', 10)),
                    'acc_x': float(record.get('accelerometerAccelerationX', 0)),
                    'acc_y': float(record.get('accelerometerAccelerationY', 0)),
                    'acc_z': float(record.get('accelerometerAccelerationZ', 0)),
                    'gyro_x': float(record.get('gyroRotationX', 0)),
                    'gyro_y': float(record.get('gyroRotationY', 0)),
                    'gyro_z': float(record.get('gyroRotationZ', 0)),
                    'motion_heading': float(record.get('motionHeading', -1)),
                    'motion_roll': float(record.get('motionRoll', 0)),
                    'motion_pitch': float(record.get('motionPitch', 0)),
                    'motion_yaw': float(record.get('motionYaw', 0))
                }
                
                # Flag valid GPS data
                data['has_valid_gps'] = (data['latitude'] != 0 and 
                                         data['longitude'] != 0 and 
                                         abs(data['latitude']) <= 90 and 
                                         abs(data['longitude']) <= 180 and 
                                         loc_timestamp > 0)
                
                records.append(data)
                
            except (KeyError, ValueError) as e:
                continue
    
    print(f"Collected {len(records)} total records")
    
    if not records:
        print("No valid records found")
        return None
    
    # Convert to DataFrame and sort by timestamp
    df = pl.DataFrame(records).sort('timestamp')
    
    # Extract valid GPS positions for initial processing
    gps_records = df.filter(pl.col('has_valid_gps') == True).to_pandas()
    
    if len(gps_records) < 2:
        print("Not enough valid GPS points for fusion")
        return None
    
    print(f"Found {len(gps_records)} valid GPS points")
    
    # Convert to pandas for processing
    all_records = df.to_pandas()
    
    # STEP 1: Create high-resolution time base for fusion
    min_time = all_records['timestamp'].min()
    max_time = all_records['timestamp'].max()
    
    # Calculate median sampling rate of accelerometer
    accel_intervals = np.diff(all_records['timestamp'].values)
    median_interval = np.median(accel_intervals)
    
    print(f"Median sensor interval: {median_interval*1000:.2f} ms")
    print(f"Approximate sensor frequency: {1/median_interval:.2f} Hz")
    
    # STEP 2: Interpolate GPS positions to high-resolution time base
    # Extract GPS data
    gps_times = gps_records['timestamp'].values
    gps_lats = gps_records['latitude'].values
    gps_lons = gps_records['longitude'].values
    
    # Create interpolation functions
    # Use cubic spline for smoother transitions
    lat_interp = interpolate.splrep(gps_times, gps_lats, s=0.001)
    lon_interp = interpolate.splrep(gps_times, gps_lons, s=0.001)
    
    # STEP 3: Enhance with accelerometer and gyroscope data
    # Initialize output data
    fused_positions = []
    
    # Process each record
    last_valid_gps_idx = 0
    
    for idx, row in all_records.iterrows():
        current_time = row['timestamp']
        
        # Interpolate GPS position for this timestamp
        if current_time >= gps_times[0] and current_time <= gps_times[-1]:
            # We're within the GPS time range, interpolate
            lat_interpolated = float(interpolate.splev(current_time, lat_interp))
            lon_interpolated = float(interpolate.splev(current_time, lon_interp))
            
            # Find nearest GPS points for accuracy estimation
            next_idx = np.searchsorted(gps_times, current_time)
            if next_idx >= len(gps_times):
                next_idx = len(gps_times) - 1
            prev_idx = max(0, next_idx - 1)
            
            # Calculate time ratios for accuracy
            t_ratio = 1.0
            if gps_times[next_idx] > gps_times[prev_idx]:
                t_ratio = (current_time - gps_times[prev_idx]) / (gps_times[next_idx] - gps_times[prev_idx])
            
            # Use a confidence metric based on distance from actual GPS points
            confidence = 1.0 - min(t_ratio, 1.0 - t_ratio) 
            
            source = "Interpolated"
        elif row['has_valid_gps']:
            # Direct GPS measurement
            lat_interpolated = row['latitude']
            lon_interpolated = row['longitude']
            confidence = 1.0
            source = "GPS"
        else:
            # Outside GPS range, use last known position
            if fused_positions:
                last_pos = fused_positions[-1]
                lat_interpolated = last_pos['latitude']
                lon_interpolated = last_pos['longitude']
                confidence = max(0.1, last_pos['confidence'] - 0.05)  # Decreasing confidence
                source = "Extrapolated"
            else:
                # No position to use
                continue
        
        # Calculate acceleration magnitude (G-forces)
        total_g = np.sqrt(row['acc_x']**2 + row['acc_y']**2 + row['acc_z']**2)
        
        # Store fused position
        fused_positions.append({
            'timestamp': current_time,
            'latitude': lat_interpolated,
            'longitude': lon_interpolated,
            'altitude': row['altitude'],
            'speed': max(0, row['speed']) if row['speed'] >= 0 else 0,
            'course': row['course'] if row['course'] >= 0 else 
                     (row['motion_heading'] if row['motion_heading'] >= 0 else 0),
            'acceleration_x': row['acc_x'],
            'acceleration_y': row['acc_y'],
            'acceleration_z': row['acc_z'],
            'total_g': total_g,
            'confidence': confidence,
            'source': source
        })
    
    # STEP 4: Apply smoothing to reduce noise
    # Convert to DataFrame
    fused_df = pd.DataFrame(fused_positions)
    
    # Apply Savitzky-Golay filter for smoothing
    # This preserves peaks better than moving average
    window_size = 11  # Must be odd
    poly_order = 3   # Polynomial order
    
    if len(fused_df) > window_size:
        fused_df['latitude_smooth'] = savgol_filter(fused_df['latitude'], window_size, poly_order)
        fused_df['longitude_smooth'] = savgol_filter(fused_df['longitude'], window_size, poly_order)
    else:
        fused_df['latitude_smooth'] = fused_df['latitude']
        fused_df['longitude_smooth'] = fused_df['longitude']
    
    # Convert back to polars for export
    result_df = pl.from_pandas(fused_df)
    
    # STEP 5: Convert timestamp to ISO format for Kepler compatibility
    result_df = result_df.with_columns(
        pl.lit("2025-09-21T").alias("date_prefix")
    )
    
    # Format as Kepler.gl compatible timestamp
    result_df = result_df.with_columns([
        (pl.col("date_prefix") + pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.strftime("%H:%M:%S")).alias("kepler_timestamp")
    ])
    
    # Write to CSV
    result_df.write_csv(output_path)
    print(f"Exported fused location data to {output_path}")
    print(f"File size: {Path(output_path).stat().st_size / (1024*1024):.2f} MB")
    
    # STEP 6: Visualize results
    # Convert to pandas for easier plotting
    fused_pd = result_df.to_pandas()
    gps_only = fused_pd[fused_pd['source'] == 'GPS']
    
    # Plot comparison of original GPS vs fused data
    plt.figure(figsize=(12, 10))
    
    # Plot GPS points
    plt.scatter(gps_only['longitude'], gps_only['latitude'], 
               color='blue', s=30, label='Original GPS', alpha=0.7)
    
    # Plot smooth path line
    plt.plot(fused_pd['longitude_smooth'], fused_pd['latitude_smooth'], 
            'r-', linewidth=1.5, label='Fused Path')
    
    plt.title('GPS vs Fused Path')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.legend()
    plt.axis('equal')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    # Plot speed vs G-force
    plt.figure(figsize=(10, 6))
    plt.scatter(fused_pd['speed'], fused_pd['total_g'], alpha=0.3, c=fused_pd['total_g'], cmap='viridis')
    plt.colorbar(label='G-Force')
    plt.title('Speed vs G-Force')
    plt.xlabel('Speed (m/s)')
    plt.ylabel('G-Force')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    return result_df

# Run the karting data fusion
fusion_output_path = '/Users/nathanverrill/Karting/COTA/cota_fused_kepler.csv'
fused_df = process_karting_data_fusion(file_path, fusion_output_path, START_TIMESTAMP_SEC, END_TIMESTAMP_SEC)

In [ ]:
def analyze_karting_performance(fused_df):
    """
    Analyze karting performance metrics from fused data
    """
    from scipy.spatial.distance import pdist, squareform
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    
    from sklearn.cluster import DBSCAN
    
    # Convert to pandas for analysis
    df = fused_df.to_pandas() if hasattr(fused_df, 'to_pandas') else fused_df
    
    print("Analyzing karting performance metrics...")
    
    # 1. Detect laps based on position
    def detect_laps(df):
        """
        Detect lap completions based on position crossing start/finish
        """
        # For COTA, we need to know start/finish coordinates
        # Simplified approach: look for points where the path crosses itself
        
        # Get latitude and longitude columns
        lats = df['latitude_smooth'].values
        lons = df['longitude_smooth'].values
        timestamps = df['timestamp'].values
        
        # Combine into coordinate array
        coords = np.column_stack([lats, lons])
        
        # Compute distances between all points
        dists = squareform(pdist(coords))
        
        # Look for points that are close in space but far in time
        min_time_diff = 60  # Minimum lap time in seconds
        proximity_threshold = 0.0001  # Proximity threshold in degrees
        
        lap_crossings = []
        
        for i in range(len(df) // 2):  # Only check first half of points
            for j in range(i + len(df) // 4, len(df)):  # Check against later points
                # If points are close in space
                if dists[i, j] < proximity_threshold:
                    # And far enough in time
                    time_diff = timestamps[j] - timestamps[i]
                    if time_diff > min_time_diff:
                        lap_crossings.append((i, j, time_diff))
        
        # Sort by time
        lap_crossings.sort(key=lambda x: x[0])
        
        # Group into laps
        laps = []
        if lap_crossings:
            start_idx = 0
            for crossing_start, crossing_end, lap_time in lap_crossings:
                laps.append({
                    'start_idx': start_idx,
                    'end_idx': crossing_end,
                    'start_time': timestamps[start_idx],
                    'end_time': timestamps[crossing_end],
                    'lap_time': timestamps[crossing_end] - timestamps[start_idx]
                })
                start_idx = crossing_end
        
        return laps
    
    # Detect laps
    laps = detect_laps(df)
    
    print(f"Detected {len(laps)} laps")
    for i, lap in enumerate(laps):
        print(f"Lap {i+1}: {lap['lap_time']:.2f} seconds")
    
    # 3. Identify braking zones
    def identify_braking_zones(df):
        """
        Identify major braking zones based on deceleration
        """
        # Calculate acceleration between points
        df['delta_speed'] = df['speed'].diff()
        df['delta_time'] = df['timestamp'].diff()
        
        # Avoid division by zero
        df['delta_time'] = df['delta_time'].replace(0, np.nan)
        
        # Calculate deceleration (negative acceleration)
        df['deceleration'] = df['delta_speed'] / df['delta_time']
        
        # Filter for significant deceleration events
        braking_threshold = -1.0  # m/s^2
        braking_points = df[df['deceleration'] < braking_threshold].copy()
        
        # Group nearby braking points into zones using clustering
        if len(braking_points) > 10:
            # Extract coordinates
            braking_coords = braking_points[['latitude_smooth', 'longitude_smooth']].values
            
            # Use DBSCAN clustering to identify distinct braking zones
            clustering = DBSCAN(eps=0.0001, min_samples=5).fit(braking_coords)
            braking_points['zone_id'] = clustering.labels_
            
            # Filter out noise points (labeled as -1)
            valid_zones = braking_points[braking_points['zone_id'] >= 0]
            
            # Get zone summary statistics
            zones = []
            for zone_id in sorted(valid_zones['zone_id'].unique()):
                zone_points = valid_zones[valid_zones['zone_id'] == zone_id]
                
                # Calculate zone centroid
                centroid_lat = zone_points['latitude_smooth'].mean()
                centroid_lon = zone_points['longitude_smooth'].mean()
                
                # Get zone statistics
                min_speed = zone_points['speed'].min()
                max_speed = zone_points['speed'].max()
                avg_decel = zone_points['deceleration'].mean()
                
                zones.append({
                    'zone_id': zone_id,
                    'latitude': centroid_lat,
                    'longitude': centroid_lon,
                    'entry_speed': max_speed,
                    'exit_speed': min_speed,
                    'speed_delta': max_speed - min_speed,
                    'avg_deceleration': avg_decel,
                    'point_count': len(zone_points)
                })
            
            return zones, braking_points
        else:
            return [], braking_points
    
    # Identify braking zones
    braking_zones, braking_points = identify_braking_zones(df)
    
    # Print braking zone information
    print(f"\nIdentified {len(braking_zones)} major braking zones:")
    for i, zone in enumerate(braking_zones):
        print(f"Braking Zone {i+1}:")
        print(f"  Location: {zone['latitude']:.6f}, {zone['longitude']:.6f}")
        print(f"  Entry Speed: {zone['entry_speed']:.1f} m/s ({zone['entry_speed']*2.237:.1f} mph)")
        print(f"  Exit Speed: {zone['exit_speed']:.1f} m/s ({zone['exit_speed']*2.237:.1f} mph)")
        print(f"  Speed Reduction: {zone['speed_delta']:.1f} m/s ({zone['speed_delta']*2.237:.1f} mph)")
        print(f"  Avg. Deceleration: {abs(zone['avg_deceleration']):.2f} m/s²")
    
    # 4. Identify cornering zones (high lateral G areas)
    def identify_cornering_zones(df):
        """
        Identify corners based on lateral acceleration and direction changes
        """
        # Assuming acceleration_y represents lateral acceleration
        # You might need to adjust based on how your device is oriented
        
        # Calculate lateral G force (absolute value)
        df['lateral_g'] = df['acceleration_y'].abs()
        
        # Threshold for cornering
        cornering_threshold = 0.5  # G
        cornering_points = df[df['lateral_g'] > cornering_threshold].copy()
        
        # Cluster cornering points
        if len(cornering_points) > 10:
            cornering_coords = cornering_points[['latitude_smooth', 'longitude_smooth']].values
            
            # Use DBSCAN clustering
            clustering = DBSCAN(eps=0.0001, min_samples=5).fit(cornering_coords)
            cornering_points['corner_id'] = clustering.labels_
            
            # Filter valid corners
            valid_corners = cornering_points[cornering_points['corner_id'] >= 0]
            
            # Get corner summary
            corners = []
            for corner_id in sorted(valid_corners['corner_id'].unique()):
                corner_points = valid_corners[valid_corners['corner_id'] == corner_id]
                
                # Calculate corner centroid
                centroid_lat = corner_points['latitude_smooth'].mean()
                centroid_lon = corner_points['longitude_smooth'].mean()
                
                # Get statistics
                max_g = corner_points['lateral_g'].max()
                avg_g = corner_points['lateral_g'].mean()
                avg_speed = corner_points['speed'].mean()
                
                corners.append({
                    'corner_id': corner_id,
                    'latitude': centroid_lat,
                    'longitude': centroid_lon,
                    'max_lateral_g': max_g,
                    'avg_lateral_g': avg_g,
                    'avg_speed': avg_speed,
                    'point_count': len(corner_points)
                })
            
            return corners, cornering_points
        else:
            return [], cornering_points
    
    # Identify cornering zones
    cornering_zones, cornering_points = identify_cornering_zones(df)
    
    # Print cornering information
    print(f"\nIdentified {len(cornering_zones)} corners:")
    for i, corner in enumerate(cornering_zones):
        print(f"Corner {i+1}:")
        print(f"  Location: {corner['latitude']:.6f}, {corner['longitude']:.6f}")
        print(f"  Max Lateral G: {corner['max_lateral_g']:.2f}")
        print(f"  Avg Speed: {corner['avg_speed']:.1f} m/s ({corner['avg_speed']*2.237:.1f} mph)")
    
    # 5. Visualize the track with braking and cornering zones
    plt.figure(figsize=(12, 10))
    
    # Plot the full track
    plt.plot(df['longitude_smooth'], df['latitude_smooth'], 'k-', linewidth=1, alpha=0.5)
    
    # Plot braking zones
    if len(braking_zones) > 0:
        braking_lats = [zone['latitude'] for zone in braking_zones]
        braking_lons = [zone['longitude'] for zone in braking_zones]
        plt.scatter(braking_lons, braking_lats, c='red', s=100, marker='s', label='Braking Zones')
        
        # Add zone numbers
        for i, (lon, lat) in enumerate(zip(braking_lons, braking_lats)):
            plt.text(lon, lat, str(i+1), fontsize=12, ha='center', va='center', color='white')
    
    # Plot cornering zones
    if len(cornering_zones) > 0:
        corner_lats = [corner['latitude'] for corner in cornering_zones]
        corner_lons = [corner['longitude'] for corner in cornering_zones]
        plt.scatter(corner_lons, corner_lats, c='blue', s=100, marker='o', label='Corners')
        
        # Add corner numbers
        for i, (lon, lat) in enumerate(zip(corner_lons, corner_lats)):
            plt.text(lon, lat, str(i+1), fontsize=12, ha='center', va='center', color='white')
    
    plt.title('Track Map with Braking Zones and Corners')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.legend()
    plt.axis('equal')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    # Return analysis results
    return {
        'laps': laps,
        'braking_zones': braking_zones,
        'cornering_zones': cornering_zones
    }

# If we have fused data, analyze it
if 'fused_df' in locals() and fused_df is not None:
    analysis_results = analyze_karting_performance(fused_df)

In [ ]:
def prepare_for_kepler(fused_df, analysis_results, output_path):
    """
    Prepare fused data for Kepler.gl with additional features
    """
    import pandas as pd
    import polars as pl
    from datetime import datetime
    
    print("Preparing data for Kepler.gl...")
    
    # Convert to pandas for processing
    df = fused_df.to_pandas() if hasattr(fused_df, 'to_pandas') else fused_df.copy()
    
    # 1. Add lap information if available
    if 'laps' in analysis_results and analysis_results['laps']:
        # Create lap column
        df['lap'] = -1
        
        for i, lap in enumerate(analysis_results['laps']):
            # Mark points in this lap
            lap_mask = (df['timestamp'] >= lap['start_time']) & (df['timestamp'] <= lap['end_time'])
            df.loc[lap_mask, 'lap'] = i + 1
    
    # 2. Add ISO format timestamp for Kepler
    # Convert Unix timestamps to datetime strings that Kepler likes
    base_date = "2025-09-21T"
    df['kepler_time'] = df['timestamp'].apply(
        lambda x: f"{base_date}{datetime.fromtimestamp(x).strftime('%H:%M:%S')}"
    )
    
    # 3. Create trip_id column for Kepler trip layers
    df['trip_id'] = df['lap'].astype(str)
    df.loc[df['lap'] == -1, 'trip_id'] = "0"  # For points not in a lap
    
    # 4. Format columns for Kepler
    kepler_df = df[['kepler_time', 'latitude_smooth', 'longitude_smooth', 
                    'speed', 'total_g', 'trip_id', 'lap', 
                    'acceleration_x', 'acceleration_y', 'acceleration_z']].copy()
    
    # Rename columns for Kepler
    kepler_df = kepler_df.rename(columns={
        'kepler_time': 'timestamp',
        'latitude_smooth': 'latitude',
        'longitude_smooth': 'longitude'
    })
    
    # 5. Convert speed to mph for easier reading in Kepler
    kepler_df['speed_mph'] = kepler_df['speed'] * 2.237
    
    # 6. Add extra columns for visualization
    # Color coding for G-force
    kepler_df['g_force_category'] = pd.cut(
        kepler_df['total_g'],
        bins=[0, 0.5, 1.0, 1.5, 2.0, float('inf')],
        labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
    )
    
    # 7. Create separate dataframes for braking zones and corners
    braking_df = None
    if 'braking_zones' in analysis_results and analysis_results['braking_zones']:
        braking_df = pd.DataFrame(analysis_results['braking_zones'])
        braking_df['type'] = 'Braking Zone'
        
        # Add zone ID for labels in Kepler
        braking_df['zone_id'] = range(1, len(braking_df) + 1)
        
        # Add formatted descriptions
        braking_df['description'] = braking_df.apply(
            lambda x: f"Braking Zone {x['zone_id']}\n"
                      f"Entry: {x['entry_speed']*2.237:.1f} mph\n"
                      f"Exit: {x['exit_speed']*2.237:.1f} mph\n"
                      f"Delta: {x['speed_delta']*2.237:.1f} mph",
            axis=1
        )
    
    corner_df = None
    if 'cornering_zones' in analysis_results and analysis_results['cornering_zones']:
        corner_df = pd.DataFrame(analysis_results['cornering_zones'])
        corner_df['type'] = 'Corner'
        
        # Add corner ID for labels in Kepler
        corner_df['corner_id'] = range(1, len(corner_df) + 1)
        
        # Add formatted descriptions
        corner_df['description'] = corner_df.apply(
            lambda x: f"Corner {x['corner_id']}\n"
                      f"Max G: {x['max_lateral_g']:.2f}\n"
                      f"Avg Speed: {x['avg_speed']*2.237:.1f} mph",
            axis=1
        )
    
    # 8. Save main track data
    kepler_df.to_csv(output_path, index=False)
    print(f"Saved main track data to {output_path}")
    
    # 9. Save zones data if available
    if braking_df is not None:
        braking_output = output_path.replace('.csv', '_braking_zones.csv')
        braking_df.to_csv(braking_output, index=False)
        print(f"Saved braking zones to {braking_output}")
    
    if corner_df is not None:
        corner_output = output_path.replace('.csv', '_corners.csv')
        corner_df.to_csv(corner_output, index=False)
        print(f"Saved corner data to {corner_output}")
    
    # 10. Create a combined POI file for easier Kepler loading
    if braking_df is not None or corner_df is not None:
        poi_dfs = []
        
        if braking_df is not None:
            poi_dfs.append(braking_df[['zone_id', 'latitude', 'longitude', 'type', 'description']]
                          .rename(columns={'zone_id': 'id'}))
        
        if corner_df is not None:
            poi_dfs.append(corner_df[['corner_id', 'latitude', 'longitude', 'type', 'description']]
                          .rename(columns={'corner_id': 'id'}))
        
        if poi_dfs:
            poi_df = pd.concat(poi_dfs)
            poi_output = output_path.replace('.csv', '_points_of_interest.csv')
            poi_df.to_csv(poi_output, index=False)
            print(f"Saved combined points of interest to {poi_output}")
    
    return kepler_df, braking_df, corner_df

# Execute export for Kepler if we have analysis results
if 'fused_df' in locals() and 'analysis_results' in locals():
    kepler_output = '/Users/nathanverrill/Karting/COTA/cota_kepler_enhanced.csv'
    kepler_df, braking_df, corner_df = prepare_for_kepler(fused_df, analysis_results, kepler_output)

In [ ]:
def process_karting_data_complete_workflow(file_path, output_dir, start_timestamp_ms, end_timestamp_ms):
    """
    Complete workflow to process karting data from JSON to Kepler.gl visualization
    
    Parameters:
    - file_path: Path to raw JSON sensor log
    - output_dir: Directory to save output files
    - start_timestamp_ms: Start timestamp in milliseconds
    - end_timestamp_ms: End timestamp in milliseconds
    """
    import polars as pl
    import ijson
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy import interpolate
    from scipy.signal import savgol_filter
    from pathlib import Path
    import os
    from datetime import datetime
    from sklearn.cluster import DBSCAN
    %matplotlib inline
    
    # Convert timestamps to seconds
    start_timestamp_sec = start_timestamp_ms / 1000.0
    end_timestamp_sec = end_timestamp_ms / 1000.0
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Processing karting data from {file_path}")
    print(f"Time range: {datetime.fromtimestamp(start_timestamp_sec)} to {datetime.fromtimestamp(end_timestamp_sec)}")
    
    # Step 1: Initial data loading and filtering
    print("\nStep 1: Loading and filtering data...")
    
    # First pass: collect all records within timestamp range
    records = []
    record_count = 0
    
    with open(file_path, 'rb') as f:
        for record in ijson.items(f, 'item'):
            record_count += 1
            
            if record_count % 10000 == 0:
                print(f"Processed {record_count} records...", end='\r')
                
            try:
                # Get timestamps
                loc_timestamp = float(record.get('locationTimestamp_since1970', 0))
                accel_timestamp = float(record.get('accelerometerTimestamp_sinceReboot', 0))
                
                # Skip records outside the timestamp range
                if loc_timestamp > 0 and (loc_timestamp < start_timestamp_sec or loc_timestamp > end_timestamp_sec):
                    continue
                
                # Get all relevant data
                data = {
                    'timestamp': accel_timestamp,
                    'gps_timestamp': loc_timestamp if loc_timestamp > 0 else None,
                    'latitude': float(record.get('locationLatitude', 0)),
                    'longitude': float(record.get('locationLongitude', 0)),
                    'altitude': float(record.get('locationAltitude', 0)),
                    'speed': float(record.get('locationSpeed', -1)),
                    'course': float(record.get('locationTrueHeading', -1)),
                    'horizontal_accuracy': float(record.get('locationHorizontalAccuracy', 10)),
                    'acc_x': float(record.get('accelerometerAccelerationX', 0)),
                    'acc_y': float(record.get('accelerometerAccelerationY', 0)),
                    'acc_z': float(record.get('accelerometerAccelerationZ', 0)),
                    'gyro_x': float(record.get('gyroRotationX', 0)),
                    'gyro_y': float(record.get('gyroRotationY', 0)),
                    'gyro_z': float(record.get('gyroRotationZ', 0)),
                    'motion_heading': float(record.get('motionHeading', -1)),
                    'motion_roll': float(record.get('motionRoll', 0)),
                    'motion_pitch': float(record.get('motionPitch', 0)),
                    'motion_yaw': float(record.get('motionYaw', 0))
                }
                
                # Flag valid GPS data
                data['has_valid_gps'] = (data['latitude'] != 0 and 
                                         data['longitude'] != 0 and 
                                         abs(data['latitude']) <= 90 and 
                                         abs(data['longitude']) <= 180 and 
                                         loc_timestamp > 0)
                
                records.append(data)
                
            except (KeyError, ValueError) as e:
                continue
    
    print(f"\nLoaded {len(records)} records from {record_count} total records")
    
    # Convert to DataFrame
    df = pl.DataFrame(records)
    
    # Save raw filtered data
    raw_output = os.path.join(output_dir, "raw_filtered_data.csv")
    df.write_csv(raw_output)
    print(f"Saved raw filtered data to {raw_output}")
    
    # Step 2: Perform sensor fusion
    print("\nStep 2: Performing sensor fusion...")
    
    # Convert to pandas for fusion processing
    df_pd = df.to_pandas()
    
    # Extract GPS records
    gps_records = df_pd[df_pd['has_valid_gps']].copy()
    print(f"Found {len(gps_records)} valid GPS records")
    
    # Check if we have enough GPS data
    if len(gps_records) < 10:
        print("Warning: Not enough GPS data for reliable fusion")
    
    # Create interpolation for GPS coordinates
    if len(gps_records) >= 2:
        # Sort GPS records by timestamp
        gps_records = gps_records.sort_values('gps_timestamp')
        
        # Get GPS data arrays
        gps_times = gps_records['gps_timestamp'].values
        gps_lats = gps_records['latitude'].values
        gps_lons = gps_records['longitude'].values
        
        # Create spline interpolation
        try:
            # Smoothness parameter - higher values mean more smoothing
            smoothness = 0.01 * len(gps_times) 
            
            # Create the interpolation functions
            lat_interp = interpolate.splrep(gps_times, gps_lats, s=smoothness)
            lon_interp = interpolate.splrep(gps_times, gps_lons, s=smoothness)
            
            # Flag to indicate successful interpolation
            interpolation_success = True
        except Exception as e:
            print(f"Error creating interpolation: {e}")
            interpolation_success = False
    else:
        interpolation_success = False
    
    # Initialize fusion results array
    fused_positions = []
    
    # Sort all records by timestamp
    df_pd = df_pd.sort_values('timestamp')
    
    # Process each record
    for idx, row in df_pd.iterrows():
        current_time = row['timestamp']
        current_gps_time = row['gps_timestamp'] if pd.notnull(row['gps_timestamp']) else None
        
        # Determine position source and coordinates
        if row['has_valid_gps']:
            # Direct GPS measurement
            lat = row['latitude']
            lon = row['longitude']
            position_source = 'GPS'
            confidence = 1.0
        elif interpolation_success and current_gps_time is not None:
            # Check if time is within interpolation range
            if gps_times[0] <= current_gps_time <= gps_times[-1]:
                # Use spline interpolation
                try:
                    lat = float(interpolate.splev(current_gps_time, lat_interp))
                    lon = float(interpolate.splev(current_gps_time, lon_interp))
                    position_source = 'Interpolated'
                    
                    # Calculate confidence based on distance to nearest GPS point
                    time_diffs = np.abs(gps_times - current_gps_time)
                    min_time_diff = np.min(time_diffs)
                    confidence = max(0.5, 1.0 - (min_time_diff / 5.0))  # Higher confidence if closer to GPS point
                except:
                    # Fall back to last valid position
                    if fused_positions:
                        last_pos = fused_positions[-1]
                        lat = last_pos['latitude']
                        lon = last_pos['longitude']
                        position_source = 'Fallback'
                        confidence = 0.3
                    else:
                        continue  # Skip if we have no position yet
            else:
                # Outside interpolation range
                if fused_positions:
                    last_pos = fused_positions[-1]
                    lat = last_pos['latitude']
                    lon = last_pos['longitude']
                    position_source = 'Extrapolated'
                    confidence = 0.2
                else:
                    continue  # Skip if we have no position yet
        else:
            # No GPS or interpolation, use last known position
            if fused_positions: